In [5]:
import pandas as pd
import numpy as np
import requests
import json
import os
import time
import random
import http.client
import logging
import traceback
from datetime import datetime
from typing import Optional

# ==================== 全局配置 ====================

API_HOST = "api.apidance.pro"
API_KEY = "q4fa83ok43io70najdgmijdt2s6fkl"

# 最多翻多少页（用户时间线）
MAX_PAGES_PER_USER = 1000

# 单个用户内，每一页请求失败时的最多重试次数
MAX_RETRIES_PER_PAGE = 3

# 获取 rest_id 时的最多重试次数
MAX_RETRIES_REST_ID = 3

# 判断是否“覆盖到 2024-01-01 及以前”
COVER_TARGET_DATE = datetime(2024, 1, 1)

# 是否保存每个用户的原始 tweets JSON
SAVE_RAW_TWEETS = False

# 路径配置
log_dir = r'I:\finance-agent\X\logs'
tweets_save_dir = r'I:\finance-agent\X\tweets_json'
summary_output_path = r'I:\finance-agent\X\kol_time_coverage.xlsx'
kol_excel_path = r'I:\finance-agent\X\a_kol.xlsx'

os.makedirs(log_dir, exist_ok=True)
os.makedirs(tweets_save_dir, exist_ok=True)

log_filename = os.path.join(
    log_dir, f'tweets_time_range_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)


# ==================== 基础接口封装 ====================

def get_user_rest_id(user_name: str) -> Optional[str]:
    """
    获取用户的 rest_id，带重试。
    """
    for attempt in range(1, MAX_RETRIES_REST_ID + 1):
        try:
            conn = http.client.HTTPSConnection(API_HOST, timeout=30)
            headers = {'apikey': API_KEY}

            url = (
                "/graphql/UserByScreenName?"
                "variables=%7B%22screen_name%22:%22"
                + user_name +
                "%22,%22withSafetyModeUserFields%22:true,%22withHighlightedLabel%22:true%7D"
            )
            conn.request("GET", url, '', headers)

            res = conn.getresponse()
            data = res.read()
            json_data = json.loads(data.decode("utf-8"))

            rest_id = json_data['data']['user']['result']['rest_id']
            logger.info(f"✓ 获取用户 @{user_name} 的 rest_id: {rest_id}")
            return rest_id

        except Exception as e:
            logger.error(f"✗ 获取用户 @{user_name} 的 rest_id 失败 (第 {attempt} 次): {e}")
            logger.debug(traceback.format_exc())
            if attempt < MAX_RETRIES_REST_ID:
                wait = 2 ** attempt + random.uniform(0.5, 2.0)
                logger.info(f"等待 {wait:.1f} 秒后重试获取 rest_id ...")
                time.sleep(wait)
            else:
                logger.error(f"多次尝试仍然无法获取 @{user_name} 的 rest_id")
                return None

    return None


def get_user_tweets_page(user_id: str, cursor: Optional[str] = None) -> dict:
    """
    获取单页用户推文（/sapi/UserTweets）。
    真正的重试逻辑放在上层 fetch_user_tweets_and_time_range 里。
    这里仅负责做一次请求。
    """
    conn = http.client.HTTPSConnection(API_HOST, timeout=30)
    headers = {'apikey': API_KEY}

    url = f"/sapi/UserTweets?user_id={user_id}"
    if cursor:
        url += f"&cursor={cursor}"
    else:
        url += "&cursor=null"

    conn.request("GET", url, '', headers)
    res = conn.getresponse()
    data = res.read()
    return json.loads(data.decode("utf-8"))


def parse_tweet_created_at(tweet: dict) -> Optional[datetime]:
    """
    从 /sapi/UserTweets 的 tweet 结构中解析 created_at:
    例如: "Sat Dec 06 12:30:00 +0000 2025"
    """
    created_at_str = tweet.get("created_at")
    if not created_at_str:
        return None
    try:
        dt = datetime.strptime(created_at_str, "%a %b %d %H:%M:%S %z %Y")
        return dt.replace(tzinfo=None)
    except Exception:
        return None


# ==================== 抓取 + 时间范围统计 ====================

def fetch_user_tweets_and_time_range(user_id: str, user_name: str, max_pages: int):
    """
    获取用户多页推文，同时统计时间范围。
    带“每页重试”逻辑：
        - 每一页请求最多重试 MAX_RETRIES_PER_PAGE 次
        - 某一页多次失败，则认为该用户抓取失败，提前结束
    返回:
        tweets: list[dict] (如果 SAVE_RAW_TWEETS=True，否则空列表)
        earliest_dt: datetime | None
        latest_dt: datetime | None
        failed_reason: str | None  如果因为多次失败中断，这里写原因
    """
    all_tweets = []
    cursor: Optional[str] = None
    page = 1

    earliest_dt: Optional[datetime] = None
    latest_dt: Optional[datetime] = None
    failed_reason: Optional[str] = None

    logger.info(f"开始获取用户 @{user_name} (ID: {user_id}) 的推文，最多 {max_pages} 页")

    while page <= max_pages:
        logger.info(f"@{user_name} 准备获取第 {page} 页 (cursor={cursor})")

        # ----- 这一页带重试 -----
        success_this_page = False
        tweets_this_page = []

        for attempt in range(1, MAX_RETRIES_PER_PAGE + 1):
            try:
                resp = get_user_tweets_page(user_id, cursor)
                tweets_this_page = resp.get("tweets", [])
                logger.info(
                    f"@{user_name} 第 {page} 页 第 {attempt} 次请求成功，"
                    f"拿到 {len(tweets_this_page)} 条 tweets"
                )
                success_this_page = True

                # 更新 cursor
                next_cursor = resp.get("next_cursor_str")
                if not next_cursor:
                    logger.info(f"@{user_name} 第 {page} 页没有 next_cursor，认为没有更多推文")
                    cursor = None
                else:
                    cursor = next_cursor

                break

            except Exception as e:
                logger.error(
                    f"@{user_name} 获取第 {page} 页失败 (第 {attempt} 次): {e}"
                )
                logger.debug(traceback.format_exc())

                if attempt < MAX_RETRIES_PER_PAGE:
                    wait = 2 ** attempt + random.uniform(0.5, 2.0)
                    logger.info(f"等待 {wait:.1f} 秒后重试该页...")
                    time.sleep(wait)
                else:
                    failed_reason = (
                        f"第 {page} 页连续 {MAX_RETRIES_PER_PAGE} 次请求失败，停止该用户"
                    )
                    logger.error(failed_reason)

        # 如果这一页多次重试仍然失败，退出整个用户
        if not success_this_page:
            break

        # ----- 正常处理这一页数据 -----
        logger.info(f"@{user_name} 第 {page} 页: 有效 tweets 数量 {len(tweets_this_page)}")

        for tw in tweets_this_page:
            dt = parse_tweet_created_at(tw)
            if dt:
                if (earliest_dt is None) or (dt < earliest_dt):
                    earliest_dt = dt
                if (latest_dt is None) or (dt > latest_dt):
                    latest_dt = dt

        if SAVE_RAW_TWEETS:
            all_tweets.extend(tweets_this_page)

        # 没有 next_cursor 说明到头了
        if not cursor:
            logger.info(f"@{user_name} 没有更多推文了，提前结束 (共 {page} 页)")
            break

        page += 1
        # 每页之间小睡一下
        time.sleep(0.4)

    logger.info(
        f"@{user_name} 抓取结束：最早时间={earliest_dt}, 最晚时间={latest_dt}, "
        f"失败原因={failed_reason}"
    )

    return all_tweets, earliest_dt, latest_dt, failed_reason


def save_tweets(user_name: str, tweets: list, save_dir: str) -> bool:
    """保存推文到文件（可选）"""
    try:
        os.makedirs(save_dir, exist_ok=True)
        filename = os.path.join(save_dir, f'{user_name}.json')

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tweets, f, ensure_ascii=False, indent=2)

        logger.info(f"✓ @{user_name} 推文已保存到 {filename}")
        return True

    except Exception as e:
        logger.error(f"✗ @{user_name} 保存失败: {e}")
        logger.debug(traceback.format_exc())
        return False


# ==================== 单用户汇总 ====================

def process_single_user_for_range(user_name: str, max_pages: int) -> dict:
    """
    对单个用户：
        - 获取 rest_id (带重试)
        - 按页抓取 tweets (每页带重试)
        - 统计时间范围
        - 返回 summary dict
    """
    logger.info("=" * 60)
    logger.info(f"开始处理用户: @{user_name}")
    logger.info("=" * 60)

    summary = {
        "user_name": user_name,
        "user_id": None,
        "earliest_created_at": None,
        "latest_created_at": None,
        "covers_2024_01_or_earlier": False,
        "remark": ""
    }

    user_id = get_user_rest_id(user_name)
    if not user_id:
        summary["remark"] = "rest_id 获取失败（多次重试）"
        logger.info(f"@{user_name} rest_id 获取失败，跳过")
        return summary

    summary["user_id"] = user_id

    tweets, earliest_dt, latest_dt, failed_reason = fetch_user_tweets_and_time_range(
        user_id, user_name, max_pages
    )

    summary["earliest_created_at"] = earliest_dt
    summary["latest_created_at"] = latest_dt

    if failed_reason:
        # 因为多次请求失败而中断
        summary["remark"] = f"抓取过程中断: {failed_reason}"
    elif earliest_dt is None and latest_dt is None:
        summary["remark"] = "未获取到任何有效 created_at（可能确实没有推文）"
    else:
        summary["covers_2024_01_or_earlier"] = bool(
            earliest_dt and earliest_dt <= COVER_TARGET_DATE
        )
        if summary["covers_2024_01_or_earlier"]:
            summary["remark"] = "覆盖到 2024-01 之前 ✅"
        else:
            summary["remark"] = "最早时间晚于 2024-01 ❌"

    if SAVE_RAW_TWEETS and tweets:
        save_tweets(user_name, tweets, tweets_save_dir)

    logger.info(
        f"@{user_name} 完成：最早={earliest_dt}, 最晚={latest_dt}, "
        f"覆盖到 2024-01={summary['covers_2024_01_or_earlier']}, remark={summary['remark']}"
    )
    logger.info("=" * 60 + "\n")

    return summary


# ==================== 主程序 ====================

if __name__ == "__main__":
    logger.info("读取 a_kol.xlsx 文件...")
    a_kol_df = pd.read_excel(kol_excel_path)
    a_kol_list = a_kol_df['Twitter Handle'].tolist()
    a_kol_list = [str(account).lstrip('@').strip() for account in a_kol_list]

    logger.info(f"共读取 {len(a_kol_list)} 个用户")

    summaries = []

    # 你也可以先只测前几个，确认没问题再开全量
    user_list_to_test = a_kol_list
    # user_list_to_test = a_kol_list[:10]

    start_time = time.time()
    for i, user in enumerate(user_list_to_test, 1):
        logger.info(f"\n进度: {i}/{len(user_list_to_test)}  (@{user})")
        summary = process_single_user_for_range(user, MAX_PAGES_PER_USER)
        summaries.append(summary)
        # 为了稳一点，每个用户之间也稍微暂停
        time.sleep(1.0)

    elapsed = time.time() - start_time
    logger.info(f"\n所有用户处理完成，总耗时 {elapsed:.2f} 秒")

    # 汇总到 DataFrame 并导出
    df_summary = pd.DataFrame(summaries)

    # datetime 转字符串
    for col in ["earliest_created_at", "latest_created_at"]:
        df_summary[col] = df_summary[col].apply(
            lambda x: x.strftime("%Y-%m-%d %H:%M:%S") if isinstance(x, datetime) else None
        )

    df_summary.to_excel(summary_output_path, index=False)
    logger.info(f"时间覆盖汇总已保存到：{summary_output_path}")


2025-12-06 20:47:26,170 [INFO] 读取 a_kol.xlsx 文件...
2025-12-06 20:47:26,215 [INFO] 共读取 199 个用户
2025-12-06 20:47:26,216 [INFO] 
进度: 1/199  (@TheEconomist)
2025-12-06 20:47:26,216 [INFO] ============================================================
2025-12-06 20:47:26,217 [INFO] 开始处理用户: @TheEconomist
2025-12-06 20:47:26,218 [INFO] ============================================================
2025-12-06 20:47:27,722 [INFO] ✓ 获取用户 @TheEconomist 的 rest_id: 5988062
2025-12-06 20:47:27,726 [INFO] 开始获取用户 @TheEconomist (ID: 5988062) 的推文，最多 1000 页
2025-12-06 20:47:27,726 [INFO] @TheEconomist 准备获取第 1 页 (cursor=None)
2025-12-06 20:47:30,018 [INFO] @TheEconomist 第 1 页 第 1 次请求成功，拿到 20 条 tweets
2025-12-06 20:47:30,019 [INFO] @TheEconomist 第 1 页: 有效 tweets 数量 20
2025-12-06 20:47:30,421 [INFO] @TheEconomist 准备获取第 2 页 (cursor=DAAHCgABG7fJvvP__-oLAAIAAAATMTk5NzIxNDQ3NzAxMjMzNzA0OAgAAwAAAAIAAA)
2025-12-06 20:47:32,803 [INFO] @TheEconomist 第 2 页 第 1 次请求成功，拿到 20 条 tweets
2025-12-06 20:47:32,804 [INFO] @TheEcon

KeyboardInterrupt: 